# 04a - Self-Play Generation

Optional RL stage. Collect the next iteration with frozen models on CPU by default. Generation does not apply gradient updates; 04b training uses CUDA when available.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Project and Dependencies

Keep Colab's existing CUDA PyTorch. Restart only if pip explicitly requires it.

In [ ]:
from pathlib import Path
import sys
import subprocess

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl").is_dir():
    raise FileNotFoundError(f"Project files are missing from {PROJECT_ROOT}. See README.md.")
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", str(PROJECT_ROOT / "requirements_colab.txt")])

## Run Configuration

Edit configs/default.yaml once for the workflow. Use a new run_id for a changed experiment.

In [ ]:
from chess_rl.config import load_config, prepare_directories
from chess_rl.reproducibility import metadata, read_json, atomic_json, sha256

prepare_directories(PROJECT_ROOT)
cfg = load_config(PROJECT_ROOT, "strategy.yaml")
print("Run:", cfg["run_id"])
print("Runtime:", metadata())

## Load Initialization

Default is the completed combined strategy checkpoint. Broad initialization requires strategy explicitly skipped. To override, set research_workflow.self_play_initial_checkpoint before beginning a new league run.

In [ ]:
from chess_rl.research_workflow import self_play_inputs
initial_checkpoint, dataset_manifest = self_play_inputs(PROJECT_ROOT, cfg)
print("Frozen initialization:", initial_checkpoint)
print("Settings:", cfg["self_play"])

## Generate or Resume Games

Records use engine-search policy targets and completed-game results. Failed games are excluded from training. Strategy validation/test positions and reserved broad/opening positions are excluded. Completed games and their hashes are persisted.

In [ ]:
from chess_rl.self_play import generate_self_play
generation = generate_self_play(PROJECT_ROOT, cfg, initial_checkpoint, dataset_manifest)
print("Generation:", generation.get("iteration", generation.get("status")))
print("Next: 04b_self_play_training_and_promotion.ipynb.")